<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/03_PCMCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the pcmci family of causal discovery algorithms on the correctedv3 dataset.
The goal of this is to show that between batches , its a memoryless process


Primary workflow levels

---



In [1]:

# Order-level features
#     ↓ split by city
# City data
#     ↓ split by courier
# Courier delivery sequence
#     ↓ PCMCI+ separately per courier
# Courier causal graphs
#     ↓ count recurring significant links
# City-level consensus graph



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
DATASET_PATH = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'

OUTPUT_DIR = "/content/drive/MyDrive/ml/CORRECTEDv3/causal_graphs/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [4]:
!pip install tigramite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.7/314.7 kB 6.5 MB/s eta 0:00:00


# Objective

- PCMCI+ — single pooled temporal causal discovery ,  Output : Directed lagged graph
- J-PCMCI+ — joint multi-city temporal causal discovery , Output : Joint graph

# why reduced set of features

- PCMCI+ runtime scales roughly as O(N² × T) for the skeleton phase and O(N³ × T) for the MCI test phase, where N is the number of variables and T is the time-series length.

- Going from N=28 to N=10 reduces the MCI phase by a factor of (28/10)³ ≈ 22×.

-  ParCorr conditions on linear combinations of the conditioning set — if we include both X and a monotone transform of X (like workload_causal and workload_capped), the conditioning sets become linearly dependent and the partial correlation tests lose meaning.

In [5]:
import pandas as pd
import numpy as np
import tigramite
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

# 1. Load the dataset
dataset_path = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'
df = pd.read_parquet(dataset_path)

---
Data is arranged according to the following variables :

city , delivery_user_id, receipt_time.

The rows are later ordered by receipt_time. Each courier's details are treated as time series

Use of **parcorr** - The theoretical backing is the faithfulness condition (Spirtes et al., 2000): PCMCI+ assumes the graph is faithful to the distribution, but including functional redundancies violates this because the conditional independences you observe are artefacts of the transforms, not of the causal structure.

- eta_mins                     ← target
- workload_causal              ← primary queue-depth cause
- batch_size                   ← dispatch volume
- batch_rank_dispatch          ← position in batch sequence  
- pickup_destination_distance  ← physical constraint
speed_mean_15m               ← pre-delivery mobility state
spatial_congestion_norm      ← area-level congestion (one version only)
WSI                          ← weather composite
hour_sin                     ← continuous time-of-day (exogenous driver)
hour_cos                     ← paired cyclical component

In [6]:
# 2. Define the features requested
# requested_features = [
#     'workload_causal', 'workload_capped', 'high_load', 'overloaded',
#     'pickup_destination_distance', 'batch_size', 'batch_rank_dispatch',
#     'batch_rank_capped', 'late_batch', 'extreme_batch', 'hour_sin', 'hour_cos',
#     'day_sin', 'day_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve',
#     'spatial_congestion_index_daily', 'spatial_congestion_index_rolling7',
#     'spatial_congestion_norm', 'courier_local_load', 'WSI', 'precipitation',
#     'temperature_2m', 'windspeed_10m', 'is_trajectory_available', 'typecode_cb', 'eta_mins'
# ]  original set

PCMCI_FEATURES = [
    "eta_mins",                      # target = order delivery duration
    "workload_causal",               # queue depth at dispatch = active courier workload at dispatch
    "batch_size",                    # orders dispatched simultaneously = order assigned together
    "batch_rank_dispatch",           # position in dispatch sequence = orders position within the batch
    "pickup_destination_distance",   # physical constraint (invariant) =
    "speed_mean_15m",                # pre-delivery mobility state = couriers recent delivery speed
    "spatial_congestion_norm",       # area-level delivery density (z-scored) =
    "WSI",                           # weather composite
    "hour_sin",                      # time-of-day: continuous cyclical
    "hour_cos",                      # paired with hour_sin
]

In [7]:
OPTIONAL = ["delivery_sequence_daily", "day_sin", "day_cos"]

MIN_OBSERVATIONS = 50   # minimum deliveries per courier to include in analysis = atleast 50 deliveries are analysed
TAU_MAX          = 3    # 3 deliveries back; For every delivery, PCMCI+ examines the current observation and three previous observations.
PC_ALPHA         = 0.01 # This controls graph discovery and candidate pruning. conservative value (standard)
ALPHA_MCI        = 0.01 # MCI test threshold , This is later used to declare links significant from p_matrix.

---

Idea is to include operational state, order complexity,mobility,environment,
temporal patterns.

**Here a lag means a delivery position, not a fixed amount of clock time:**

In [8]:
def run_pcmci_for_city(city_name: str, df: pd.DataFrame) -> dict:
    """
    Run PCMCI+ per courier within a city, then aggregate results.

    Each courier is treated as an independent realisation of the same
    data-generating process. Running per-courier avoids cross-courier
    spurious lag links while allowing aggregation via edge frequency.

    Returns
    -------
    dict  {courier_id: pcmci_result} for couriers with sufficient observations
    """
    # Use only available features
    features = [f for f in PCMCI_FEATURES if f in df.columns]
    features += [f for f in OPTIONAL if f in df.columns]

    # Structural fill for GPS nulls before analysis (consistent with add_gps_missingness_flag design)
    # Missing GPS-derived values are replaced with zero. This encodes: missing GPS = zero speed or zero movement
    df = df.copy()
    for col in ["speed_mean_15m", "speed_std_15m", "distance_travelled_15m"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Remaining nulls in selected features — fill with column mean
    # (weather ASOF join occasionally misses boundary rows)
    df[features] = df[features].fillna(df[features].mean())


    # TODO : #### POTENTIAL PROBLEM

    ## Each missing value is replaced with the city-wide feature mean.
# This prevents ParCorr from failing, but introduces two problems:
# It uses future observations while imputing earlier observations.
# It artificially reduces variance and correlations around missing records.



# ----------------------------------------------------------------------------------------------------


    couriers = df["delivery_user_id"].unique()
    results  = {}
    parcorr  = ParCorr(significance="analytic")

# Constructing over each courier

    for courier_id in couriers:
        courier_df = (
            df[df["delivery_user_id"] == courier_id]
            .sort_values("receipt_time")
            [features]
        )

        if len(courier_df) < MIN_OBSERVATIONS:
            continue  # insufficient time-series length for tau_max=3 , skip


        data_array  = courier_df.values.astype(float)
        tg_dataframe = pp.DataFrame(
            data_array,
            var_names=features,
            datatime=np.arange(len(courier_df)),
        )

        # Tigramite sees deliveries as regularly spaced time steps, even though receipt times may be irregular.

        pcmci = PCMCI(
            dataframe=tg_dataframe,
            cond_ind_test=parcorr,
            verbosity=0,  # suppress per-courier output
        )

        result = pcmci.run_pcmciplus(
            tau_min=0,       # include contemporaneous (tau=0) links
            tau_max=TAU_MAX,
            pc_alpha=PC_ALPHA,
        )


        results[courier_id] = result

    print(f"  {city_name}: {len(results)} couriers with ≥{MIN_OBSERVATIONS} obs")
    return results, features

# -------------------------------------------





Results of run_pcmci_per_city contains 3 things

- result["graph"]
- result["p_matrix"]
- result["val_matrix"]



---


**graph **:
Contains PCMCI+ edge marks and orientations:
- -->    directed
- <--    reverse-directed
- o-o    unresolved orientation
- x-x    conflicting orientation
- ""     no edge



---

**p_matrix** : Contains conditional-independence p-values.


Its interpretation is: p_matrix[source, target, lag]


Example: p_matrix[i, j, 2] tests the candidate relationship:
feature_i[t-2] → feature_j[t]




---


**val_matrix** = Contains the ParCorr test statistic.
 It indicates: sign of association & strength of conditional association.



In [9]:
def aggregate_edge_frequency(
    city_results: dict,
    features: list,
    alpha_level: float = ALPHA_MCI,
) -> np.ndarray:
    """
    Compute edge presence frequency across couriers (bootstrap consensus).

    For each edge (i, j, tau), count what fraction of couriers show
    a significant link. This is the P(e) bootstrap stability score from
    Stage 5 of the thesis pipeline.

    Returns
    -------
    np.ndarray  shape (N, N, tau_max+1) — edge frequency matrix
    """
    N       = len(features)
    freq    = np.zeros((N, N, TAU_MAX + 1))
    n_total = len(city_results)

    if n_total == 0:
        return freq

    for result in city_results.values():
        p_matrix = result["p_matrix"]  # shape (N, N, tau_max+1)
        significant = (p_matrix <= alpha_level).astype(float)
        freq += significant

    return freq / n_total  # proportion of couriers with significant edge


    # returns courier selection frequency.
    # Suppose 80 out of 100 couriers have: workload[t-1] → eta[t] , The frequency becomes: freq[workload, eta, 1] = 0.80


In [ ]:
# ── Run per city ──────────────────────────────────────────────────────────────
city_graphs   = {}
city_features = {}

# Group the existing 'df' by city instead of looking for external files
for city_name, df_city in df.groupby('city'):
    print(f"\n{'='*55}")
    print(f"  PCMCI+ — {city_name}")
    print(f"{'='*55}")

    # Run the analysis using the grouped dataframe
    results, feat_list = run_pcmci_for_city(city_name, df_city)

    freq_matrix = aggregate_edge_frequency(results, feat_list)
    city_graphs[city_name]   = freq_matrix
    city_features[city_name] = feat_list

    # Save per-city results
    city_slug = city_name.lower().replace(' ', '_')
    np.save(f"{OUTPUT_DIR}/pcmci_freq_{city_slug}.npy", freq_matrix)
    np.save(f"{OUTPUT_DIR}/pcmci_features_{city_slug}.npy", np.array(feat_list))
    print(f"  Saved frequency matrix: shape {freq_matrix.shape}")


# ── Print consensus edges (appear in >50% of couriers) ───────────────────────
CONSENSUS_THRESHOLD = 0.50

for city_name, freq in city_graphs.items():
    feat = city_features[city_name]
    print(f"\n--- Consensus edges (P(e) > {CONSENSUS_THRESHOLD}) — {city_name} ---")
    N = len(feat)
    found = False
    for i in range(N):
        for j in range(N):
            for tau in range(TAU_MAX + 1):
                if freq[i, j, tau] > CONSENSUS_THRESHOLD:
                    direction = "→" if tau > 0 else "↔"
                    lag_str   = f"(lag {tau})" if tau > 0 else "(contemp.)"
                    print(f"  {feat[i]} {direction} {feat[j]}  {lag_str}"
                          f"  P(e) = {freq[i,j,tau]:.2f}")
                    found = True
    if not found:
        print("  No consensus edges at this threshold.")


  PCMCI+ — Chongqing
  Chongqing: 119 couriers with ≥50 obs
  Saved frequency matrix: shape (12, 12, 4)

  PCMCI+ — Hangzhou


In [ ]:
import matplotlib.pyplot as plt
from tigramite import plotting as tp
import numpy as np

# Visualize results for cities that processed successfully
for city_name, freq in city_graphs.items():
    feat = city_features[city_name]
    print(f"\nGraph for {city_name} (Consensus > {CONSENSUS_THRESHOLD}):")

    # Create a boolean graph mask where frequency exceeds threshold
    # Tigramite expects a string array or boolean array for the 'graph' argument
    # '' for no edge, '-->' for directed, 'o-o' for undirected/contemporaneous
    N = len(feat)
    graph_mask = np.zeros(freq.shape, dtype='<U3')

    for i in range(N):
        for j in range(N):
            for tau in range(TAU_MAX + 1):
                if freq[i, j, tau] > CONSENSUS_THRESHOLD:
                    if tau == 0:
                        graph_mask[i, j, tau] = 'o-o'
                    else:
                        graph_mask[i, j, tau] = '-->'

    # Plotting contemporaneous and lagged links
    tp.plot_graph(
        graph=graph_mask,
        val_matrix=freq,
        var_names=feat,
        show_colorbar=True,
        node_size=0.5,
        arrow_linewidth=2.0,
        label_fontsize=10
    )
    plt.show()

### Understanding the Output

The output displays the results of the **PCMCI+ algorithm**, which identifies causal relationships between different features of your delivery dataset.

#### 1. Consensus Edges (P(e) > 0.5)
- **P(e) (Bootstrap Stability):** This represents the fraction of couriers for whom this specific link was statistically significant. A value of `0.98` means that 98% of the couriers in that city exhibited this relationship.
- **Lagged Edges (→):** A link like `workload_causal → workload_causal (lag 1)` indicates **auto-correlation**. It means the workload in the previous delivery step strongly predicts the workload in the next step.
- **Contemporaneous Edges (↔):** These are relationships that happen within the same time step. For example, `batch_size ↔ batch_rank_dispatch` indicates they are inherently tied together (as one increases, the other usually does too within the same dispatch event).

#### 2. Causal Graphs
In the generated plots:
- **Nodes:** Represents features (e.g., `eta_mins`, `batch_size`).
- **Directed Arrows (-->):** Represent a causal effect from the past to the future (lagged effect).
- **Non-directed Lines (o-o):** Represent instantaneous (contemporaneous) correlations where the direction of causality couldn't be determined within that single time step.
- **Colors/Thickness:** Usually represent the strength of the relationship (based on the frequency matrix `freq`).

#### Key Findings in the Data:
* **System Persistence:** Features like `workload_causal`, `WSI` (weather), and `day_sin/cos` show very high stability (`P(e) ~ 1.0`), meaning they are consistent drivers across almost all couriers.
* **Dispatch Logic:** The near-perfect link between `batch_size` and `batch_rank_dispatch` confirms the mechanical relationship in how orders are assigned.
* **ETA Drivers:** In Shanghai, we can see `pickup_destination_distance ↔ eta_mins`, confirming that physical distance is a primary contemporaneous driver of delivery time.

### Batch-Level PCMCI+ Analysis
In this section, we test the hypothesis that the delivery process is memoryless between batches. We aggregate order-level features into batch-level averages and run PCMCI+ to check for cross-batch dependencies.

In [ ]:
def run_batch_pcmci(city_name: str, df: pd.DataFrame):
    # 1. Aggregate to batch level
    # We group by courier and batch ID to ensure we don't mix couriers during aggregation
    batch_df = df.groupby(['delivery_user_id', 'from_dipan_id']).agg({
        'receipt_time': 'min',
        'eta_mins': 'mean',
        'workload_causal': 'mean',
        'batch_size': 'first', # batch_size is invariant within a batch
        'pickup_destination_distance': 'mean',
        'speed_mean_15m': 'mean',
        'spatial_congestion_norm': 'mean',
        'WSI': 'mean',
        'hour_sin': 'first',
        'hour_cos': 'first'
    }).reset_index()

    # Sort by courier and time to maintain sequence
    batch_df = batch_df.sort_values(['delivery_user_id', 'receipt_time'])

    features = ["eta_mins", "workload_causal", "batch_size", "pickup_destination_distance", "speed_mean_15m", "WSI"]

    couriers = batch_df['delivery_user_id'].unique()
    batch_results = {}
    parcorr = ParCorr(significance='analytic')

    for courier_id in couriers:
        c_df = batch_df[batch_df['delivery_user_id'] == courier_id][features]

        # Lowered observation threshold for batches since couriers have fewer batches than orders
        if len(c_df) < 30:
            continue

        data_array = c_df.values.astype(float)
        tg_df = pp.DataFrame(data_array, var_names=features)

        pcmci = PCMCI(dataframe=tg_df, cond_ind_test=parcorr, verbosity=0)
        results = pcmci.run_pcmciplus(tau_min=0, tau_max=2, pc_alpha=0.01)
        batch_results[courier_id] = results

    return batch_results, features

# Run for a specific city as a representative test (e.g., Chongqing)
chongqing_df = df[df['city'] == 'Chongqing']
batch_results, batch_feat = run_batch_pcmci('Chongqing', chongqing_df)
batch_freq = aggregate_edge_frequency(batch_results, batch_feat)

print(f"Batch-level analysis complete for Chongqing ({len(batch_results)} couriers analysed).")

In [ ]:
# Visualize Batch-Level Consensus
print("\n--- Batch-Level Consensus Edges (P(e) > 0.4) ---")
N_b = len(batch_feat)
for i in range(N_b):
    for j in range(N_b):
        for tau in range(1, 3): # Focus on Lags to test 'memoryless' property
            if batch_freq[i, j, tau] > 0.4:
                print(f"  [LAGGED] {batch_feat[i]} (t-{tau}) → {batch_feat[j]} (t)  P(e) = {batch_freq[i,j,tau]:.2f}")

# Contemporaneous links for comparison
for i in range(N_b):
    for j in range(i+1, N_b):
        if batch_freq[i, j, 0] > 0.4:
            print(f"  [CONTEMP] {batch_feat[i]} ↔ {batch_feat[j]}  P(e) = {batch_freq[i,j,0]:.2f}")

---

JPCMCI+



In [ ]:
!pip install tigramite
import pandas as pd
import numpy as np
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.jpcmciplus import JPCMCIplus
from tigramite.independence_tests.parcorr import ParCorr
from pathlib import Path

# ── Config ──
CITY_CODES = {"Shanghai": 0, "Hangzhou": 1, "Chongqing": 2}
PCMCI_FEATURES = [
    "eta_mins", "workload_causal", "batch_size", "batch_rank_dispatch",
    "pickup_destination_distance", "speed_mean_15m", "spatial_congestion_norm",
    "WSI", "hour_sin", "hour_cos"
]
MIN_OBSERVATIONS = 50
TAU_MAX = 3
PC_ALPHA = 0.01
DATASET_PATH = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'

def prepare_city_data(city_name: str, path: str, features: list) -> pd.DataFrame:
    df_full = pd.read_parquet(path)
    df_city = df_full[df_full['city'] == city_name].copy()
    df_city = df_city.sort_values(["delivery_user_id", "receipt_time"]).reset_index(drop=True)
    gps_cols = ["speed_mean_15m", "speed_std_15m", "distance_travelled_15m"]
    for col in gps_cols:
        if col in df_city.columns: df_city[col] = df_city[col].fillna(0)
    df_city[features] = df_city[features].fillna(df_city[features].mean())
    df_city["_city_code"] = CITY_CODES[city_name]
    return df_city

def build_jpcmci_dataset(city_dfs: dict, features: list):
    segments, mask_segs = [], []
    for city_name, df_c in city_dfs.items():
        for courier_id, cdf in df_c.groupby("delivery_user_id"):
            if len(cdf) < MIN_OBSERVATIONS: continue
            block = cdf[features + ["_city_code"]].values.astype(float)
            segments.append(block)
            block_mask = np.zeros((len(block), len(features) + 1), dtype=bool)
            block_mask[:TAU_MAX, :] = True
            mask_segs.append(block_mask)
    return np.vstack(segments), np.vstack(mask_segs), features + ["city_context"]

# ── Execution ──
features_to_use = [f for f in PCMCI_FEATURES if f in df.columns]
city_data = {name: prepare_city_data(name, DATASET_PATH, features_to_use) for name in CITY_CODES.keys()}
data_arr, mask_arr, var_names = build_jpcmci_dataset(city_data, features_to_use)
tg_df = pp.DataFrame(data_arr, mask=mask_arr, var_names=var_names)

N = len(var_names)
node_classification = {i: 0 for i in range(N-1)}
node_classification[N-1] = 2

# Initialize JPCMCIplus
jpcmci = JPCMCIplus(
    dataframe=tg_df,
    cond_ind_test=ParCorr(significance='analytic'),
    node_classification=node_classification
)

# Manually define link assumptions and initialize parent sets to prevent KeyError
link_assumptions = {j: {(i, -tau): 'o?o' for i in range(N) for tau in range(TAU_MAX+1) if not (tau==0 and i==j)} for j in range(N)}

# Run the discovery
results_j = jpcmci.run_jpcmciplus(
    link_assumptions=link_assumptions,
    tau_max=TAU_MAX,
    pc_alpha=PC_ALPHA
)

jpcmci.print_significant_links(p_matrix=results_j['p_matrix'], val_matrix=results_j['val_matrix'], alpha_level=PC_ALPHA)